In [1]:
import random
!pip install -U scikit-learn pandas numpy matplotlib seaborn scipy

^C


In [2]:
import pandas as pd
import numpy as np



In [3]:
def load_data(path: str):
    df = pd.read_csv(path)
    return df

data = load_data("./data/Patients Data.csv")
data.columns.tolist()

['PatientID',
 'State',
 'Sex',
 'GeneralHealth',
 'AgeCategory',
 'HeightInMeters',
 'WeightInKilograms',
 'BMI',
 'HadHeartAttack',
 'HadAngina',
 'HadStroke',
 'HadAsthma',
 'HadSkinCancer',
 'HadCOPD',
 'HadDepressiveDisorder',
 'HadKidneyDisease',
 'HadArthritis',
 'HadDiabetes',
 'DeafOrHardOfHearing',
 'BlindOrVisionDifficulty',
 'DifficultyConcentrating',
 'DifficultyWalking',
 'DifficultyDressingBathing',
 'DifficultyErrands',
 'SmokerStatus',
 'ECigaretteUsage',
 'ChestScan',
 'RaceEthnicityCategory',
 'AlcoholDrinkers',
 'HIVTesting',
 'FluVaxLast12',
 'PneumoVaxEver',
 'TetanusLast10Tdap',
 'HighRiskLastYear',
 'CovidPos']

In [4]:
data.head()

,PatientID,State,Sex,GeneralHealth,AgeCategory,HeightInMeters,WeightInKilograms,BMI,HadHeartAttack,HadAngina,...,ECigaretteUsage,ChestScan,RaceEthnicityCategory,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,1,Alabama,Female,Fair,Age 75 to 79,1.63,84.820000,32.099998,0,1,...,Never used e-cigarettes in my entire life,1,"White only, Non-Hispanic",0,0,0,1,"No, did not receive any tetanus shot in the pa...",0,1
1,2,Alabama,Female,Very good,Age 65 to 69,1.60,71.669998,27.990000,0,0,...,Never used e-cigarettes in my entire life,0,"White only, Non-Hispanic",0,0,1,1,"Yes, received Tdap",0,0
2,3,Alabama,Male,Excellent,Age 60 to 64,1.78,71.209999,22.530001,0,0,...,Never used e-cigarettes in my entire life,0,"White only, Non-Hispanic",1,0,0,0,"Yes, received tetanus shot but not sure what type",0,0
3,4,Alabama,Male,Very good,Age 70 to 74,1.78,95.250000,30.129999,0,0,...,Never used e-cigarettes in my entire life,0,"White only, Non-Hispanic",0,0,1,1,"Yes, received tetanus shot but not sure what type",0,0
4,5,Alabama,Female,Good,Age 50 to 54,1.68,78.019997,27.760000,0,0,...,Never used e-cigarettes in my entire life,1,"Black only, Non-Hispanic",0,0,1,0,"No, did not receive any tetanus shot in the pa...",0,0


In [5]:
data["AgeCategory"].value_counts()

AgeCategory
Age 65 to 69       27547
Age 60 to 64       25685
Age 70 to 74       24946
Age 55 to 59       21422
Age 50 to 54       19154
Age 75 to 79       17679
Age 80 or older    17544
Age 40 to 44       16228
Age 45 to 49       16095
Age 35 to 39       14982
Age 30 to 34       12825
Age 18 to 24       12777
Age 25 to 29       10746
Name: count, dtype: int64

In [6]:
data["State"].value_counts()

State
Washington              14241
Maryland                 8817
Minnesota                8712
Ohio                     8700
New York                 8625
Texas                    7267
Florida                  7124
Kansas                   6000
Wisconsin                5890
Maine                    5709
Iowa                     5492
Indiana                  5393
South Carolina           5360
Virginia                 5358
Arizona                  5302
Hawaii                   5262
Utah                     5212
Michigan                 5206
Massachusetts            5164
Nebraska                 5008
Colorado                 4973
Georgia                  4860
California               4801
Connecticut              4765
Vermont                  4569
South Dakota             4280
Montana                  4155
Missouri                 4042
New Jersey               3833
New Hampshire            3564
Puerto Rico              3550
Idaho                    3394
Alaska                   3100
Rhod

In [7]:
def select_features(df: pd.DataFrame, features: list):
    return df[features]

filtered_data_columns = select_features(data, ["PatientID","Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"])

In [8]:
def preprocess_data(df: pd.DataFrame):
    df.loc[:,"AgeCategory"]=df["AgeCategory"].apply(lambda x: int(random.randrange(
        int(x.strip().split(" ")[1]),
         100 if (x.strip().split(" ")[3] == 'older') else int(x.strip().split(" ")[3])
    )))

    return df
preprocessed_data = preprocess_data(filtered_data_columns)
preprocessed_data



,PatientID,Sex,WeightInKilograms,HeightInMeters,AgeCategory
0,1,Female,84.820000,1.63,76
1,2,Female,71.669998,1.60,66
2,3,Male,71.209999,1.78,63
3,4,Male,95.250000,1.78,73
4,5,Female,78.019997,1.68,50
...,...,...,...,...,...
237625,237626,Female,90.720001,1.57,63
237626,237627,Female,72.570000,1.70,55
237627,237628,Male,70.309998,1.75,45
237628,237629,Female,46.720001,1.57,27


In [9]:
def schnider_model(age: int, weight: float, height: float, sex: str) -> dict:
    sex = sex.lower()
    if sex not in ("male", "female"):
        raise ValueError("sex must be 'male' or 'female'")

    # --- Lean Body Mass (LBM) ---
    if sex == "male":
        lbm = 1.10 * weight - 128 * (weight ** 2) / (height ** 2)
    else:
        lbm = 1.07 * weight - 148 * (weight ** 2) / (height ** 2)

    # --- Compartment volumes (L) ---
    V1 = 4.27
    V2 = 18.9 - 0.391 * (age - 53)
    V3 = 238.0

    # --- Rate constants (min-1) ---
    k10 = 0.443 + 0.0107 * (weight - 77) - 0.0159 * (lbm - 59) + 0.0062 * (height - 177)
    k12 = 0.302 - 0.0056 * (age - 53)
    k13 = 0.196
    k21 = (1.29 - 0.024 * (age - 53)) / V2
    k31 = 0.0035
    ke0 = 0.456

    # --- State-space matrices ---
    # State: [C1, C2, C3, Ce]
    A = np.array([
        [-(k10 + k12 + k13),  k21,  k31,   0   ],
        [        k12,         -k21,   0,    0   ],
        [        k13,          0,   -k31,   0   ],
        [        ke0,          0,    0,   -ke0  ]
    ])

    # Input: u(t) in µg/min → enters central compartment (C1 = x1/V1)
    B = np.array([[1 / V1], [0], [0], [0]])

    # Output: effect concentration Ce (4th state)
    C = np.array([[0, 0, 0, 1]])

    return {
        "LBM":  round(lbm, 4),
        "V1":   V1,
        "V2":   round(V2, 4),
        "V3":   V3,
        "k10":  round(k10, 6),
        "k12":  round(k12, 6),
        "k13":  k13,
        "k21":  round(k21, 6),
        "k31":  k31,
        "ke0":  ke0,
        "A":    A,
        "B":    B,
        "C":    C,
    }
def generate_schnider_dataset(df: pd.DataFrame):
    params_df = pd.DataFrame(
        [
            schnider_model(
                age=row["AgeCategory"],
                weight=row["WeightInKilograms"],
                height=row["HeightInMeters"],
                sex=row["Sex"])
            for _, row in df.iterrows()
        ]
    )
    return pd.concat([df.reset_index(drop=True), params_df], axis=1)